In [1]:
# TẠO DỮ LIỆU TỔNG HỢP
import numpy as np
import pandas as pd

# Cố định random seed
np.random.seed(42)

# 1. Sinh 100000 điểm dữ liệu trải đều các trường hợp
N = 100000
t_raw = np.random.uniform(-20, 100, size=N)
h_raw = np.random.uniform(-20, 120, size=N)

df = pd.DataFrame({'temp': t_raw, 'humidity': h_raw})
df['label'] = 0

# 2. Áp dụng luật phân loại MECE
# Màng lọc 1: Dữ liệu lỗi
mask_error = (df['temp'] < 0) | (df['temp'] > 80) | (df['humidity'] < 20) | (df['humidity'] > 100)

# Màng lọc 2: Độ ẩm cao, nguy cơ nấm mốc
mask_mold = (~mask_error) & (df['humidity'] >= 70) # ~mask_error để chỉ đánh dấu những điểm dữ liệu không bị đánh dấu là lỗi

# Màng lọc 3: Nhiệt độ cao, nguy cơ cháy nổ
mask_fire = (~mask_error) & (~mask_mold) & (df['temp'] >= 38)

# Màng lọc 4: Bình thường
mask_normal = (~mask_error) & (~mask_mold) & (~mask_fire)

# Gán nhãn vào dataframe
df.loc[mask_error, 'label'] = 1
df.loc[mask_mold, 'label'] = 2
df.loc[mask_fire, 'label'] = 3
df.loc[mask_normal, 'label'] = 0

# 3. Undersampling để cân bằng nhãn, tránh bias
samples_per_class = 1000
df_final = pd.DataFrame()

for i in range(4):
  # Lấy ngẫu nhiên 250 mẫu cho mỗi nhãn
  class_subset = df[df['label'] == i].sample(n=samples_per_class, random_state=42)
  df_final = pd.concat([df_final, class_subset])

# 4. Suffle dữ liệu và lưu
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_csv("sensor_data.csv", index=False, header=False)
print("Đã tạo thành công 1000 mẫu dữ liệu MECE và lưu vào sensor_data.csv")

Đã tạo thành công 1000 mẫu dữ liệu MECE và lưu vào sensor_data.csv


In [2]:
#HUẤN LUYỆN MÔ HÌNH
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
PREFIX = "dht20_anomaly_model"

# 1. Tải dữ liệu
data = pd.read_csv("sensor_data.csv", names=["temp", "humidity", "label"])
X = data[["temp", "humidity"]].values
y = data["label"].values

X = X / 100.0

# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Xây dựng mô hình Mạng nơ-ron (3 hidden layer, 4 class)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax')
])

# Dùng sparse_categorical_crossentropy vì nhãn đang là số nguyên (0, 1, 2, 3)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

# 3. Huấn luyện mô hình
print("--- ĐANG HUẤN LUYỆN MÔ HÌNH ---")
model.fit(X_train, y_train, epochs=300, batch_size=16, validation_data=(X_test, y_test), verbose=1)
model.save(PREFIX + '.h5')

# 4. Đánh giá
print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Độ chính xác (Accuracy): {accuracy * 100:.2f}%\n")

# Dự đoán nhãn cho tập Test
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1) # Lấy class có xác suất cao nhất

# In báo cáo chi tiết
class_names = ['0: Bình thường', '1: Lỗi Cảm biến', '2: Ẩm/Nấm mốc', '3: Quá nhiệt']
print(classification_report(y_test, y_pred, target_names=class_names))

--- ĐANG HUẤN LUYỆN MÔ HÌNH ---
Epoch 1/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5325 - loss: 1.1809 - val_accuracy: 0.6500 - val_loss: 0.9432
Epoch 2/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7622 - loss: 0.7792 - val_accuracy: 0.8300 - val_loss: 0.6238
Epoch 3/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8331 - loss: 0.5088 - val_accuracy: 0.8575 - val_loss: 0.3989
Epoch 4/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8756 - loss: 0.3375 - val_accuracy: 0.9150 - val_loss: 0.2561
Epoch 5/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9159 - loss: 0.2494 - val_accuracy: 0.9125 - val_loss: 0.2423
Epoch 6/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9325 - loss: 0.2008 - val_accuracy: 0.9550 - val_loss: 0.1714
Epoch 7/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9528 - loss: 0.1649 - val_accuracy: 0.9575 - val_loss: 0.1499
Epoch 8/300
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 


--- ĐÁNH GIÁ TRÊN TẬP TEST ---
Độ chính xác (Accuracy): 99.00%

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
                 precision    recall  f1-score   support

 0: Bình thường       0.99      0.98      0.99       195
1: Lỗi Cảm biến       1.00      1.00      1.00       203
   2: Quá nhiệt       0.98      1.00      0.99       210
  3: Ẩm/Nấm mốc       0.99      0.98      0.99       192

       accuracy                           0.99       800
      macro avg       0.99      0.99      0.99       800
   weighted avg       0.99      0.99      0.99       800



In [3]:
# CONVERT MÔ HÌNH SANG TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
#converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Lưu file .tflite dạng nhị phân
with open(PREFIX + ".tflite", "wb") as f:
    f.write(tflite_model)

# Chuyển đổi thành C Array cho ESP32
tflite_path = PREFIX + '.tflite'
output_header_path = PREFIX + '.h'

with open(tflite_path, 'rb') as tflite_file:
    tflite_content = tflite_file.read()

# Chuyển các byte thành định dạng hex (0x..)
hex_lines = [', '.join([f'0x{byte:02x}' for byte in tflite_content[i:i + 12]])
             for i in range(0, len(tflite_content), 12)]

hex_array = ',\n  '.join(hex_lines)

# Ghi vào file Header (.h)
with open(output_header_path, 'w') as header_file:
    header_file.write('alignas(16) const unsigned char dht20_anomaly_model_tflite[] = {\n  ')
    header_file.write(f'{hex_array}\n')
    header_file.write('};\n\n')

print(f"Đã xuất model thành công ra file: {output_header_path}")
print(f"Kích thước model: {len(tflite_content)} bytes")

Saved artifact at '/tmp/tmpigd0mwfk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  132827217054992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217056144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217054800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217053456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217056720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217053648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217057104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132827217055952: TensorSpec(shape=(), dtype=tf.resource, name=None)
Đã xuất model thành công ra file: dht20_anomaly_model.h
Kích thước model: 13968 bytes
